In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Resolve project root directory (handles running from inside notebooks/)
BASE_DIR = Path.cwd()
if BASE_DIR.name == "notebooks":
    BASE_DIR = BASE_DIR.parent

# Define data directories
DATA_RAW_DIR = BASE_DIR / "data" / "raw"
DATA_PROCESSED_DIR = BASE_DIR / "data" / "processed"
DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Set input and output file paths
INPUT_PATH = DATA_RAW_DIR / "Divar.csv"
OUTPUT_PATH = DATA_PROCESSED_DIR / "preprocessing_output_v1.csv"

# Add project root to Python path for internal module imports
sys.path.append(str(BASE_DIR))

print(f"Input:  {INPUT_PATH}")
print(f"Output: {OUTPUT_PATH}")

In [ ]:
# Import data cleaning pipeline and utility functions
from source.financial_cleaning import (
    clean_financial_pipeline,
    split_datasets,
    add_business_flags,
    add_deal_type,
    add_full_credit_equivalent,
    clean_text_columns,
    drop_dead_columns,
    remove_impossible_values,
    winsorize_targets,
)

# Import feature engineering pipelines
from source.financial_feature_engineering import feature_engineering_pipeline

In [ ]:
# Verify input file existence and load the dataset
if not INPUT_PATH.exists():
    raise FileNotFoundError(f"Input file not found: {INPUT_PATH}")

print("Loading dataframe...")
df_raw = pd.read_csv(INPUT_PATH, low_memory=False)
print(f"Loaded successfully. Shape: {df_raw.shape}")

# Create a copy to preserve raw data during processing
df = df_raw.copy()

# --- Step 1: Cleaning outliers and impossible values ---
print("\n--- Step 1: Cleaning outliers and impossible values ---")
pipeline_output = clean_financial_pipeline(df)

# Handle pipeline output (unpack if returned as a tuple)
if isinstance(pipeline_output, tuple):
    df_step1 = pipeline_output[0]
else:
    df_step1 = pipeline_output

# --- Step 2: Feature Engineering ---
print("\n--- Step 2: Feature Engineering ---")
df_cleaned = feature_engineering_pipeline(df_step1)

print(f"\n Preprocessing completed. Cleaned Shape: {df_cleaned.shape}")

### Column Count Summary:

- **Initial raw columns:** `61`
- **Step 1 (Cleaning):**
  - Dropped **11** dead/redundant columns:  
    `['rent_to_single', 'rent_type', 'rent_price_on_regular_days', 'rent_price_on_special_days', 'rent_price_at_weekends', 'transformable_price', 'transformable_credit', 'transformable_rent', 'transformed_credit', 'transformed_rent', 'rent_credit_transform']`
  - Added **6** business/flag features via cleaning helper functions  
  - *Intermediate total:* $61 - 11 + 6 = 56$ columns
- **Step 2 (Feature Engineering):**
  - Added **4** financial features:  
    `['equivalent_full_credit', 'log_price_value', 'log_rent_value', 'log_credit_value']`
  - *Final total:* $56 + 4 = 60$ columns

**Final Output Shape:** `(1,000,000, 60)`


In [ ]:
def export_final_csv(df: pd.DataFrame, csv_path: Path):
    """
    Exports the cleaned dataframe to a CSV file.
    """
    print(f"Saving CSV to: {csv_path}...")

    # Ensure target directory exists
    csv_path.parent.mkdir(parents=True, exist_ok=True)

    # Export dataset with utf-8-sig encoding for Persian character support
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")

    # Calculate and display exported file size
    file_size_mb = csv_path.stat().st_size / (1024 * 1024)
    print(f"CSV saved successfully: {csv_path.name}")
    print(f"File size: {file_size_mb:.2f} MB")


# Execute export pipeline
export_final_csv(df_cleaned, OUTPUT_PATH)

# Display final dataset summary
print(f"\nFinal Report:")
print(f"   - Total Rows: {df_cleaned.shape[0]:,}")
print(f"   - Total Columns: {df_cleaned.shape[1]}")